In [1]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

import torch
import torch.nn 
import torch.nn.functional
from typing import Union, List

from ultralytics.models.rtdetr import train

batch_size = 64
context_len = 256

device = "cuda" if torch.cuda.is_available() else "cpu"

# Architettura e Implementazione di un Transformer (Decoder-only)

## 1. Rappresentazione del Dato e Pre-processing
L'obiettivo primario è la mappatura di un corpus testuale in uno spazio vettoriale computabile dal modello.

* **Tokenizzazione a livello di Carattere**: Identificazione del set di caratteri univoci (vocabolario) per definire la dimensionalità dello spazio di input.
* **Encoding Bidirezionale**: Creazione di funzioni di mapping (`token_to_id` e `id_to_token`) per tradurre i token in indici interi e viceversa.
* **Configurazione degli Iperparametri**: Definizione della dimensione dei batch (parallelismo) e della lunghezza del contesto (finestra di attenzione), parametri che condizionano la memoria e la capacità computazionale.
* **Suddivisione del Dataset**: Partizionamento tra training e validation set per garantire una corretta valutazione della capacità di generalizzazione.

PS: Nella `token_to_it` usa `"".join`

In [2]:
with open("../data/shakespeare.txt", "r") as f:
    raw_data = f.read()


In [3]:
tokens = sorted(set(raw_data))
voc_size = len(tokens)

In [4]:
# mapping
token_to_id = {el: i for i, el in enumerate(tokens)}
id_to_token = {i: el for i, el in enumerate(tokens)}

In [5]:
# def encoding and decoding
encode = lambda x: torch.tensor([token_to_id[s] for s in x])
decod = lambda x: "".join([id_to_token[s] for s in x])

In [6]:
# encoding data
encoded_data = encode(raw_data)

In [7]:
len_data = len(encoded_data)

train_data = encoded_data[:int(0.8 * len_data)]
val_data = encoded_data[int(0.8 * len_data): int(0.9 * len_data)]

In [8]:
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data

    #select a starting point
    ix = torch.randint(len(data) - context_len, (batch_size,))

    #select starting point + context_len
    x = torch.stack([data[i:i+context_len] for i in ix])

    #select target
    y = torch.stack([data[i+1:i+context_len+1] for i in ix])

    x, y = x.to(device), y.to(device)

    # x -> [batch_size, context_len]; y -> [batch_size, context_len] 1-hot encoded
    return x, y

In [9]:
get_batch("train")

(tensor([[57,  1, 54,  ...,  1, 58, 46],
         [46, 43, 47,  ..., 50, 53, 60],
         [52, 42,  1,  ..., 42, 43, 56],
         ...,
         [43, 45, 43,  ...,  1, 58, 46],
         [24, 27, 33,  ..., 47, 58, 46],
         [57, 53, 50,  ..., 53, 52, 45]], device='cuda:0'),
 tensor([[ 1, 54, 47,  ..., 58, 46, 43],
         [43, 47, 56,  ..., 53, 60, 43],
         [42,  1, 57,  ..., 43, 56,  1],
         ...,
         [45, 43, 58,  ..., 58, 46, 43],
         [27, 33, 15,  ..., 58, 46, 43],
         [53, 50, 42,  ..., 52, 45,  1]], device='cuda:0'))

## 3. Scaled Dot-Product Attention
Il cuore del Transformer risiede nella gestione della focalizzazione del modello sulle parti rilevanti della sequenza tramite la classe `Head`.

$$
 \text{Attention}(Q, K, V) = \text{Softmax}\left(\frac{QK^T}{\sqrt{d_k}} \right)V
$$

* **Proiezioni Lineari (Q, K, V)**: Trasformazione dell'input in vettori *Query*, *Key* e *Value* tramite matrici di peso dedicate.
* **Calcolo della Matrice di Affinità**: Prodotto scalare tra $Q$ e $K$, normalizzato per la radice quadrata della dimensione della testa per mantenere la stabilità dei gradienti.
* **Causal Masking**: Applicazione di una maschera triangolare inferiore per forzare l'attenzione a ignorare i token futuri, rispettando il vincolo di causalità del decoder.
* **Distribuzione di Probabilità**: Utilizzo della funzione Softmax per pesare l'influenza dei diversi token sulla rappresentazione finale.

## 4. Modularità e Composizione: Il Transformer Block
Integrazione di componenti modulari per facilitare l'apprendimento di feature complesse.

* **Multi-Head Attention**: Implementazione di più teste di attenzione in parallelo (es. 6 teste) per catturare simultaneamente diverse relazioni semantiche.
* **Feed-Forward Network (FFN)**: Stadio di elaborazione locale che applica trasformazioni lineari e non lineari (ReLU) ad ogni posizione della sequenza indipendentemente.
* **Stabilità Numerica**: Integrazione di *Layer Normalization* e connessioni residue (*Residual Connections*) per mitigare il problema della scomparsa del gradiente in architetture profonde.

## 5. Architettura Globale e Generazione
Integrazione finale dei moduli per la creazione del modello completo.

* **Embedding Combinato**: Somma di *Token Embedding* e *Positional Embedding* per fornire al modello sia l'identità del carattere che la sua coordinata spaziale nella sequenza.
* **Deep Stacking**: Utilizzo di una sequenza di blocchi Transformer (es. 8 blocchi) per incrementare la profondità della rappresentazione.
* **Output Head**: Proiezione lineare finale verso lo spazio del vocabolario per la generazione dei logit.
* **Inference Autoregressiva**: Generazione di nuovo testo partendo da un seed iniziale, campionando iterativamente dalla distribuzione di probabilità predetta.

PS: Ricorda dropout in HEAD nel prodotto qk e in multi head, `p = 0.2 `

PS: Module list per la multi head, poi torch.cat

PS: Ricorda la projection alla fine di tutto per riportare da emb_len a n_tokens

## Transoformers stuff



In [10]:
class Head(torch.nn.Module):
    def __init__(self, embedding_len, head_size):
        super().__init__()

        self.query = torch.nn.Linear(embedding_len, head_size, bias=False)
        self.key = torch.nn.Linear(embedding_len, head_size, bias=False)
        self.value = torch.nn.Linear(embedding_len, head_size, bias=False)

        self.register_buffer("tril", torch.tril(torch.ones(context_len, context_len))) # per salvare la triluogine come costante
        self.dropout = torch.nn.Dropout(p=0.2)
        self.sm = torch.nn.Softmax(dim=-1)
    
    def forward(self, x):
        # input x [batch_size, context_len, embedding_len]

        B, T, C = x.shape
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        d = k.shape[-1]

        qk = q @ k.transpose(-2, -1) * (d ** -0.5)
        qk = qk.masked_fill(self.tril[:T, :T] == 0, float("-inf")) # per avere matrice attenzione triangolare inferiore
        qk = self.sm(qk)

        qk = self.dropout(qk)

        return qk @ v

class MultiHeadAttention(torch.nn.Module):
    def __init__(self, embedding_len, num_heads, head_size):
        super().__init__()
        self.heads = torch.nn.ModuleList([Head(embedding_len, head_size) for _ in range(num_heads)])
        self.dropout = torch.nn.Dropout(p=0.2)

    def forward(self,x):
        out = torch.cat([head(x) for head in self.heads], dim=-1)
        out = self.dropout(out)
        return out


class FeedForward(torch.nn.Module):
    # ff part of the transformer
    def __init__(self, embedding_len):
        super().__init__()
        self.L1 = torch.nn.Linear(embedding_len, 4 * embedding_len)
        self.L2 = torch.nn.Linear(4 * embedding_len, embedding_len)

        self.r1 = torch.nn.ReLU()
        self.r1 = torch.nn.ReLU()

    def forward(self,x):
        x = self.L1(x)
        x = self.r1(x)
        x = self.L2(x)

        return x

class Block(torch.nn.Module):
    def __init__(self, embedding_len, num_heads):
        super().__init__()
        assert embedding_len % num_heads == 0, "For this example embedding len and num_heads should be multiple"
        self.ma = MultiHeadAttention(embedding_len, num_heads, embedding_len // num_heads)
        self.ln1 = torch.nn.LayerNorm(embedding_len)
        self.ff = FeedForward(embedding_len)
        self.ln2 = torch.nn.LayerNorm(embedding_len)

    def forward(self,x):
        x = self.ln1(x)
        x = x + self.ma(x)

        x = self.ln2(x)
        x = x + self.ff(x)
        return x


class PicoGPT(torch.nn.Module):
    def __init__(self, embedding_len, num_heads, num_blocks, vocab_size, context_len):
        # embeddings and positional embeddings
        super().__init__()
        self.token_emb_table = torch.nn.Embedding(vocab_size, embedding_len)
        self.pos_emb_table = torch.nn.Embedding(context_len, embedding_len)
        self.blocks = torch.nn.Sequential(*[Block(embedding_len, num_heads) for _ in range(num_blocks)]) # usando op. unpack (*)
        self.ln = torch.nn.LayerNorm(embedding_len)
        self.projection = torch.nn.Linear(embedding_len, vocab_size) # catturare gli embedding e vedere a quale posizione appartiene

        # use unpack operator to get the thing unpacked in the args

    
    def forward(self,data):
        B, T = data.shape
        data_idx = torch.arange(T).to(device)

        # retrieve the context len
        x = self.token_emb_table(data) + self.pos_emb_table(data_idx)
        x = self.blocks(x)

        return self.projection(self.ln(x))


        

In [11]:
embedding_len = 256
num_heads = 4
num_blocks = 4
num_iters = 3000

model = PicoGPT(embedding_len=embedding_len, num_heads=num_heads, num_blocks=num_blocks, vocab_size=voc_size, context_len=context_len)
loss = torch.nn.CrossEntropyLoss()
optim = torch.optim.AdamW(model.parameters(), lr = 1e-3)



In [12]:
# Impostazioni per il training
accumulation_steps = 8  # Batch size fisico(8) * step(8) = Batch size virtuale(64)
every = 100
loss_vals = []

# Inizializzazione dello scaler per l'AMP (gestisce i gradienti in mezza precisione)
scaler = torch.amp.GradScaler('cuda')

model.to(device)
model.train()

# Azzeramento dei gradienti all'inizio
optim.zero_grad(set_to_none=True)

for i in range(num_iters):
    x, y = get_batch("train")

    # 1. FORWARD PASS in Mixed Precision (AMP)
    with torch.autocast(device_type='cuda', dtype=torch.float16):
        logits = model(x)
        B, T, C = logits.shape

        loss_val = loss(logits.view(B*T, C), y.view(B*T))

        # 2. GRADIENT ACCUMULATION: Dividiamo la loss per il numero di step
        loss_val = loss_val / accumulation_steps

    # 3. BACKWARD PASS usando lo scaler per l'AMP
    scaler.scale(loss_val).backward()

    # 4. AGGIORNAMENTO PESI solo quando abbiamo accumulato abbastanza gradienti
    if (i + 1) % accumulation_steps == 0:
        scaler.step(optim)
        scaler.update()
        optim.zero_grad(set_to_none=True)

    # 5. LOGGING: Moltiplichiamo per riavere il valore "reale" della loss per il print
    if i % every == 0:
        real_loss = loss_val.item() * accumulation_steps
        loss_vals.append(real_loss)
        print(f"Loss @ iteration {i} : {real_loss:.4f}")

Loss @ iteration 0 : 4.3713
Loss @ iteration 100 : 3.1703
Loss @ iteration 200 : 2.7526
Loss @ iteration 300 : 2.5712
Loss @ iteration 400 : 2.5319
Loss @ iteration 500 : 2.4868
Loss @ iteration 600 : 2.4876
Loss @ iteration 700 : 2.4409
Loss @ iteration 800 : 2.4450
Loss @ iteration 900 : 2.4388
Loss @ iteration 1000 : 2.3769
Loss @ iteration 1100 : 2.3977
Loss @ iteration 1200 : 2.3805
Loss @ iteration 1300 : 2.3298
Loss @ iteration 1400 : 2.3325
Loss @ iteration 1500 : 2.2845
Loss @ iteration 1600 : 2.3057
Loss @ iteration 1700 : 2.2043
Loss @ iteration 1800 : 2.1790
Loss @ iteration 1900 : 2.1321
Loss @ iteration 2000 : 2.0975
Loss @ iteration 2100 : 2.0336
Loss @ iteration 2200 : 2.0403
Loss @ iteration 2300 : 2.0373
Loss @ iteration 2400 : 1.9962
Loss @ iteration 2500 : 1.9455
Loss @ iteration 2600 : 1.9100
Loss @ iteration 2700 : 1.8893
Loss @ iteration 2800 : 1.8841
Loss @ iteration 2900 : 1.8346


In [13]:
seq = "a"
# Usiamo unsqueeze per farlo diventare di shape [1, 1] anziché [1]
seq_token = encode(seq).clone().detach().unsqueeze(0).to(device)

In [15]:
sf = torch.nn.Softmax(dim = -1)
model.eval()

# seq_token.shape[1] per sapere la lunghezza, len() sul batch dà 1)
for i in range(seq_token.shape[1] - 1, 200):
    logits = model(seq_token)

    # next_token è già un tensore [1, 1] su CUDA
    next_token = torch.multinomial(sf(logits[:,-1,:]), 1)

    # Concateno il token corrente con il token prossimo
    seq_token = torch.cat((seq_token, next_token), dim=1)

In [16]:
seq_token.shape

torch.Size([1, 201])

In [18]:
print(decod(seq_token[0].tolist()))

an:
Undo I were
That brout jebair us: the I ramed to that im:
Or Duke with you, our I, bessevoid youry deatem!
Havenick's give from
Thinfe me the net's droneess frights 'To wittill his love.
Comen:
Ung
